In [1]:
import re
import random
from datasets import load_dataset, Dataset, DatasetDict
from transformers import T5Tokenizer
import pandas as pd
from typing import List, Dict, Optional
import nltk
from nltk.tokenize import sent_tokenize
import requests
import json
import numpy as np
import torch
from datasets import load_from_disk
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)


In [2]:
def create_unpunctuated_text(text: str) -> str:
    """Remove punctuation and capitalization from text"""
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def add_extra_spaces(text: str) -> str:
    """Add extra spaces randomly"""
    words = text.split()
    result = []
    for word in words:
        result.append(word)
        if random.random() < 0.3:  # 30% chance to add extra space
            result.append(' ')
    return ' '.join(result)

def remove_some_spaces( text: str) -> str:
    """Remove some spaces randomly"""
    words = text.split()
    result = []
    for i, word in enumerate(words):
        result.append(word)
        if i < len(words) - 1 and random.random() < 0.2:  # 20% chance to remove space
            continue
        else:
            result.append(' ')
    return ''.join(result).strip()

def mix_case_randomly(text: str) -> str:
    """Randomly mix uppercase and lowercase"""
    return ''.join(c.upper() if random.random() < 0.1 else c.lower() for c in text)


def load_wikipedia_dataset(language: str = "en", max_samples: Optional[int] = 100000):
    nltk.download('punkt')
    nltk.download('punkt_tab')
    try:
        wiki_dataset = load_dataset("wikimedia/wikipedia", f"20231101.{language}", split="train", streaming=True)

        samples = []
        count = 0

        for example in wiki_dataset:
            if max_samples and count >= max_samples:
                break

            text = example['text']
            sentences = sent_tokenize(text)

            for sentence in sentences:
                if len(sentence.split()) > 5 and len(sentence) < 500:
                    # Apply transformations sequentially
                    input_text = create_unpunctuated_text(sentence)
                    input_text = add_extra_spaces(input_text)
                    input_text = remove_some_spaces(input_text)
                    input_text = mix_case_randomly(input_text)

                    if input_text.strip() and input_text != sentence:
                        samples.append({
                            "input_text": f"normalize: {input_text}",
                            "target_text": sentence
                        })
                        count += 1

            if count % 1000 == 0:
                print(f"Processed {count} samples...")

        dataset = Dataset.from_list(samples)
        print(f" Created Wikipedia dataset with {len(dataset)} samples")
        return dataset

    except Exception as e:
        print(f"Error loading Wikipedia dataset: {e}")
        return None


wiki_dataset = load_wikipedia_dataset(max_samples=10000)
train_test_split = wiki_dataset.train_test_split(test_size=0.1, seed=42)

print(f"\nFinal dataset statistics:")
print(f"Training samples: {len(train_test_split['train'])}")
print(f"Validation samples: {len(train_test_split['test'])}")

# Save datasets
train_test_split['train'].save_to_disk("./large_text_normalization_train")
train_test_split['test'].save_to_disk("./large_text_normalization_val")

# Show sample
print("\nSample from dataset:")
for i in range(3):
    sample = train_test_split['train'][i]
    print(f"Input:  {sample['input_text']}")
    print(f"Output: {sample['target_text']}")
    print()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

 Created Wikipedia dataset with 10231 samples

Final dataset statistics:
Training samples: 9207
Validation samples: 1024


Saving the dataset (0/1 shards):   0%|          | 0/9207 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1024 [00:00<?, ? examples/s]


Sample from dataset:
Input:  normalize: Thiswas espEcialLy BeneFicialto theleadershipafter the intErnational1973 oilcrisis
Output: This was especially beneficial to the leadership after the international 1973 oil crisis.

Input:  normalize: aristotlesemPIricismexperiEnce and mechanics in the 4th centuRybc parmenides Publishing
Output: Aristotle's Empiricism: Experience and Mechanics in the 4th century BC, Parmenides Publishing, .

Input:  normalize: anumber of caUses arebElieved to be involveDincludinG habitat destRuctionandmodificAtion overexPlOitaTIonpollutiOn introDuced spEciEs Global warming endocrinedisRupTing pollutants dEstructionoF THe ozone layer ultraviolet radIation hAs shown to be esPecially damaging tO The skin eyes and eggs of amphibiansand diseases like chytridiomycosis
Output: A number of causes are believed to be involved, including habitat destruction and modification, over-exploitation, pollution, introduced species, global warming, endocrine-disrupting pollutants, 

In [3]:

train_dataset = load_from_disk("/content/large_text_normalization_train")
test_dataset  = load_from_disk("/content/large_text_normalization_val")


model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


def preprocess(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(inputs,
                             truncation=True, padding="max_length")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets,
                           truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset  = test_dataset.map(preprocess, batched=True)


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode
    preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    refs = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Exact string match accuracy
    correct = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
    acc = correct / len(preds)

    return {"accuracy": acc}


training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-text-normalization",
    #evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none"
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

print("Evaluating on test set...")
results = trainer.evaluate()
print(results)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

...

Map:   0%|          | 0/1024 [00:00<?, ? examples/s]

/tmp/ipython-input-892659301.py:80: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
100,1.589200
200,1.126100
300,1.036700
400,0.941600
500,0.920700
600,0.868300
700,0.818800
800,0.839300
900,0.758100
1000,0.757900


Evaluating on test set...


{'eval_loss': 0.4738459587097168, 'eval_accuracy': 0.048828125, 'eval_runtime': 82.5718, 'eval_samples_per_second': 12.401, 'eval_steps_per_second': 1.55, 'epoch': 3.0}


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import textwrap

# Load model
model_path = "/content/t5-text-normalization/checkpoint-3453/"   # adjust to your path
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Input transcript
raw_transcript = """
so yesterday i went to the university library i was trying to find some books about deep learning but then i met my friend sarah she told me hey did you start working on the project for professor smith because the deadline is next friday but honestly i had no idea it was due so soon after that we grabbed some coffee at the campus cafe and we talked about how difficult the assignments have been this semester especially the one about convolutional neural networks i said i barely managed to finish it on time and she laughed and said yeah me too i was up until 3 am working on it then we started discussing about internships and how competitive the applications are this year especially with all the companies reducing their positions because of budget cuts later in the evening we went to the gym for a quick workout but it was so crowded that we decided to just go for a walk instead the weather was nice a little windy but still good for walking we ended up talking for almost two hours without realizing the time then i rushed back home cooked some pasta watched a bit of netflix and finally started working on my database project which by the way is still only half done and due next monday so yeah it was a pretty busy day
"""

# Prefix "normalize:" if your training used that
input_text = "normalize: " + raw_transcript.strip()

# Tokenize input
inputs = tokenizer(input_text, return_tensors="pt", truncation=True)

# Generate output
with torch.no_grad():
    generated_ids = model.generate(
    **inputs,
    max_length=1024,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.9
)

chunks = textwrap.wrap(raw_transcript, 400)
normalized_chunks = []
for ch in chunks:
    input_text = "normalize: " + ch
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True)
    output_ids = model.generate(**inputs, max_length=256, num_beams=5)
    normalized_chunks.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

final_output = " ".join(normalized_chunks)
print(final_output)

print("===== RAW TRANSCRIPT =====")
print(raw_transcript)
print("\n===== NORMALIZED OUTPUT =====")
print(final_output)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


So yesterday I went to the university library, I was trying to find some books about deep learning, but then I met my friend Sarah, she told me, Hey, did you start working on the project for Professor Smith because the deadline is next Friday, but honestly, I had no idea it was due so soon after that we grabbed some coffee at the campus cafe and we talked about how difficult the assignments have been. This semester, especially the one about convolutional neural networks, I said I barely managed to finish it on time, and she laughed and said "Yeah, me too, I was up until 3 am working on it", then we started discussing about internships and how competitive the applications are this year, especially with all the companies reducing their positions because of budget cuts. Later in the evening, we went to the conference. Gym for a quick workout but it was so crowded that we decided to just go for a walk instead the weather was nice, a little windy but still good for walking, we ended up talk